In [34]:
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.linear_model import Perceptron, LogisticRegression, SGDClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, PowerTransformer

from src.auxiliares import dataframe_coeficientes
from src.config import DADOS_LIMPOS
from src.graficos import plot_coeficientes, plot_comparar_metricas_modelos
from src.models import (
    RANDOM_STATE, grid_search_cv_classificador, 
    treinar_e_validar_modelo_classificacao, organiza_resultados
)

In [35]:
df = pd.read_parquet(DADOS_LIMPOS)

df.head()

,area_mean,area_se,area_worst,compactness_mean,compactness_se,compactness_worst,concave points_mean,concave points_se,concave points_worst,concavity_mean,...,radius_worst,smoothness_mean,smoothness_se,smoothness_worst,symmetry_mean,symmetry_se,symmetry_worst,texture_mean,texture_se,texture_worst
0,1001.0,153.40,2019.0,0.27760,0.04904,0.6656,0.14710,0.01587,0.2654,0.3001,...,25.38,0.11840,0.006399,0.1622,0.2419,0.03003,0.4601,10.38,0.9053,17.33
1,1326.0,74.08,1956.0,0.07864,0.01308,0.1866,0.07017,0.01340,0.1860,0.0869,...,24.99,0.08474,0.005225,0.1238,0.1812,0.01389,0.2750,17.77,0.7339,23.41
2,1203.0,94.03,1709.0,0.15990,0.04006,0.4245,0.12790,0.02058,0.2430,0.1974,...,23.57,0.10960,0.006150,0.1444,0.2069,0.02250,0.3613,21.25,0.7869,25.53
3,386.1,27.23,567.7,0.28390,0.07458,0.8663,0.10520,0.01867,0.2575,0.2414,...,14.91,0.14250,0.009110,0.2098,0.2597,0.05963,0.6638,20.38,1.1560,26.50
4,1297.0,94.44,1575.0,0.13280,0.02461,0.2050,0.10430,0.01885,0.1625,0.1980,...,22.54,0.10030,0.011490,0.1374,0.1809,0.01756,0.2364,14.34,0.7813,16.67


In [36]:
X = df.drop(columns=['diagnosis'])

y = df['diagnosis']

In [37]:
le = LabelEncoder()

y = le.fit_transform(y)

y[:20]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0])

### seletores

In [38]:
seletor_chi2 = SelectKBest(chi2, k=15)

X_chi2 = seletor_chi2.fit_transform(X, y)

X_chi2.shape

(569, 15)

In [39]:
# ver quais colunas ele selecionou

seletor_chi2.get_feature_names_out()

array(['area_mean', 'area_se', 'area_worst', 'compactness_worst',
       'concave points_worst', 'concavity_mean', 'concavity_worst',
       'perimeter_mean', 'perimeter_se', 'perimeter_worst', 'radius_mean',
       'radius_se', 'radius_worst', 'texture_mean', 'texture_worst'],
      dtype=object)

Vamos colocar esse tratamento no pipeline

In [40]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [41]:
preprocessamento = Pipeline(steps=[
    ('scaler', PowerTransformer()),
    ('selector', SelectKBest(mutual_info_classif, k=15))
])

In [42]:
classificadores = {
    "DummyClassifier": {
        "preprocessor": None,
        "classificador": DummyClassifier(strategy='stratified')
    },
    "Perceptron": {
        "preprocessor": preprocessamento,
        "classificador": Perceptron()
    },
    "LogisticRegression": {
        "preprocessor": preprocessamento,
        "classificador": LogisticRegression()
    },
    "SGDClassifier": {
        "preprocessor": preprocessamento,
        "classificador": SGDClassifier()
    },
}

In [43]:
resultados = {
    nome_modelo: treinar_e_validar_modelo_classificacao(X, y, kf, **classificador)
    for nome_modelo, classificador in classificadores.items()
}

df_resultados = organiza_resultados(resultados)
df_resultados

,model,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,time_seconds
0,DummyClassifier,0.0025,0.015007,0.587719,0.56813,0.47191,0.456522,0.488372,0.489191,0.372331,0.017507
1,DummyClassifier,0.002003,0.017467,0.517544,0.493449,0.382022,0.369565,0.395349,0.49132,0.373237,0.01947
2,DummyClassifier,0.000998,0.013962,0.535088,0.498016,0.361446,0.365854,0.357143,0.516865,0.376735,0.01496
3,DummyClassifier,0.000997,0.018628,0.508772,0.472222,0.333333,0.333333,0.333333,0.444444,0.347063,0.019625
4,DummyClassifier,0.001995,0.016887,0.522124,0.493293,0.372093,0.363636,0.380952,0.53387,0.389054,0.018882
5,Perceptron,0.210708,0.014699,0.938596,0.918605,0.911392,1.0,0.837209,0.987881,0.982831,0.225407
6,Perceptron,0.141362,0.015621,0.894737,0.865051,0.842105,0.969697,0.744186,0.985915,0.977328,0.156983
7,Perceptron,0.138053,0.015005,0.921053,0.922619,0.896552,0.866667,0.928571,0.982143,0.978283,0.153058
8,Perceptron,0.134367,0.0,0.95614,0.960317,0.942529,0.911111,0.97619,0.995701,0.993515,0.134367
9,Perceptron,0.15015,0.0,0.964602,0.952381,0.95,1.0,0.904762,0.996982,0.995172,0.15015


In [44]:
df_resultados.groupby("model").mean().sort_values("test_recall")

,fit_time,score_time,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_average_precision,time_seconds
model,,,,,,,,,,
DummyClassifier,0.001699,0.01639,0.534249,0.505022,0.384161,0.377782,0.39103,0.495138,0.371684,0.018089
Perceptron,0.154928,0.009065,0.935026,0.923795,0.908516,0.949495,0.878184,0.989724,0.985426,0.163993
SGDClassifier,0.15786,0.01482,0.938519,0.931971,0.917563,0.933685,0.905648,0.986551,0.983466,0.17268
LogisticRegression,0.17879,0.018057,0.952585,0.947818,0.935613,0.944606,0.929125,0.992078,0.989293,0.196847


## GridSearch

In [45]:
param_grid = {
    "clf__C": [0.1, 1,  10, 100],
    "clf__penalty": ['l1', 'l2', 'elasticnet', None],
    "clf__class_weight": [None, "balanced"],
    "clf__l1_ratio" : [0.1, 0.25, 0.5, 0.75, 0.9]
}

In [46]:
clf = LogisticRegression(solver="saga", random_state=RANDOM_STATE)

grid_search = grid_search_cv_classificador(
    clf,
    param_grid, kf, preprocessamento, refit_metric='recall'
)
grid_search

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        Pipeline(steps=[('scaler',
                                                         PowerTransformer()),
                                                        ('selector',
                                                         SelectKBest(k=15,
                                                                     score_func=<function mutual_info_classif at 0x000001ED0FB32C00>))])),
                                       ('clf',
                                        LogisticRegression(random_state=42,
                                                           solver='saga'))]),
             n_jobs=-1,
             param_grid={'clf__C': [0.1, 1, 10, 100],
                         'clf__class_weight': [None, 'balanced'],
                         'clf__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9],
                         'clf__penalty': ['l1', 'l2', 'elasticnet', None]},
             refit='recall',
             scoring=['accuracy', 'balanced_accuracy', 'f1', 'precision',
                      'recall', 'roc_auc', 'average_precision'],
             verbose=1)

In [47]:
grid_search.fit(X, y)

Fitting 5 folds for each of 160 candidates, totalling 800 fits


c:\Users\breno\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\breno\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        Pipeline(steps=[('scaler',
                                                         PowerTransformer()),
                                                        ('selector',
                                                         SelectKBest(k=15,
                                                                     score_func=<function mutual_info_classif at 0x000001ED0FB32C00>))])),
                                       ('clf',
                                        LogisticRegression(random_state=42,
                                                           solver='saga'))]),
             n_jobs=-1,
             param_grid={'clf__C': [0.1, 1, 10, 100],
                         'clf__class_weight': [None, 'balanced'],
                         'clf__l1_ratio': [0.1, 0.25, 0.5, 0.75, 0.9],
                         'clf__penalty': ['l1', 'l2', 'elasticnet', None]},
             refit='recall',
             scoring=['accuracy', 'balanced_accuracy', 'f1', 'precision',
                      'recall', 'roc_auc', 'average_precision'],
             verbose=1)

In [48]:
grid_search.best_params_

{'clf__C': 1,
 'clf__class_weight': 'balanced',
 'clf__l1_ratio': 0.1,
 'clf__penalty': 'l2'}

In [49]:
grid_search.best_score_

0.9527131782945737

In [50]:
grid_search.best_estimator_['clf'].coef_

array([[-0.32498429,  1.18476725,  1.42458284, -1.4449298 ,  0.28088439,
         1.32090236,  1.53174115,  0.04005868,  1.22411142, -0.4153277 ,
        -0.37079042,  0.9207011 , -0.4775338 ,  0.54755454,  1.22321311]])